**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/nakamuta-2025"], check=True)
    os.chdir("/content/encore/entries/nakamuta-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Nakamuta et al. (2025) from dandiset 002020

**Paper.** "Self-organization of high-dimensional geometry of neural activity in culture",
bioRxiv doi:10.1101/2025.01.14.630383. The claims file
(`claims/002020/10.1101-2025.01.14.630383.yaml`) was written from version 1 of the preprint,
flattened from the bioRxiv JATS XML (Results, legends, Discussion, Methods; the supplement and the
equations of the Methods are not in that text).

**Dataset.** [DANDI:002020](https://dandiarchive.org/dandiset/002020/0.261004.1410), version
`0.261004.1410`. The paper cites no version of the dataset. This is the only published version
(published 2026-10-04); the draft has the same 129 assets and size, so the most recent published
version is pinned. It holds 129 NWB files, one 30-minute recording each, from seven dishes
(subjects), with unsorted channel-level spike detections and electrode positions.

**What can be tested.** The files hold recording dates, not days in vitro (DIV): every file says
"Days in vitro are unknown". Claims whose result is a DIV are blocked
(`metadata_insufficient`). Claims that only compare early and late recordings use DIV = days
since the dish's first recording + d0, with d0 = 2 as an assumption (d0 = 1, 3, 4 and 5 and two
per-dish readings, O2 and O3, are also shown; see "Days in vitro" below). With the PC rank as x,
the fitted exponential decay rate b of the variance spectra has a median of 0.0055 (S1, F1) and
is at or above 0.1 in 2 of 122 recordings under F1 (both near silent) and 6 under F2 (four near
silent), where 76% of the values sit at the bound 0, so the paper's 0.1 threshold between its
Exponential and Power-law groups does not separate early from late recordings here; claims that rest on
that threshold for the PCA b are blocked (`methods_underspecified`), with their results shown, and
the fitted quantity is asked of a reader (`questions/002020.yaml`). The readings of the local
Moran's I and of the locality slope give different outcomes, so the locality claims are blocked
(`methods_underspecified`), as the claims file says for readings that disagree.

| Claim | Tested |
|---|---|
| dishes-analysed | tested |
| activity-onset | blocked, metadata_insufficient (no DIV) |
| fig1b-async-to-bursts | tested, assumption on d0 |
| fig1d-power-law-emerges | blocked, methods_underspecified (b scale) |
| fig1e-example-dish | blocked, methods_underspecified (b scale; no DIV) |
| fig1f-transition-15div | blocked, methods_underspecified (b scale; no DIV) |
| fig1f-exponent-1p2 | blocked, depends on fig1f-transition-15div |
| fig1d-shuffle | blocked, methods_underspecified (b scale) |
| fig2b-spatial-structure | blocked, depends on fig2c-locality-spectrum |
| fig2c-locality-spectrum | blocked, methods_underspecified (readings disagree) |
| fig2d-locality-slope | blocked, depends on fig2c-locality-spectrum |
| fig2d-shuffle-locality | blocked, depends on fig2c-locality-spectrum |
| fig3c-avalanches-emerge | tested, fails whatever the DIV offset |
| fig3d-decay-rates-correlated | tested |
| fig3ef-groups-cooccur | blocked, methods_underspecified (b scale) |
| figs1-classified-by-avalanches | blocked, methods_underspecified (b scale) |
| figs2-shuffle-breaks-cooccurrence | blocked, methods_underspecified (readings disagree) |
| no-temporal-structure | tested, assumption on d0 |
| fig4d-cauchy-model-power-law | blocked, methods_underspecified (model not specified) |
| fig4d-gaussian-model-no-power-law | blocked, methods_underspecified (model not specified) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import resource
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import culture as C

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 40)
print("dandiset", C.DANDISET_ID, "version", C.DANDISET_VERSION)

## Loading

Every file is streamed and reduced as it loads (`culture.analyse`): 100 ms counts, the
variance spectrum of the original and the shuffled counts, the locality of every PC, cascade size
distributions, synchrony and PC time scales, under the selection readings S1 and S2. Three
processes each hold one file at a time.

In [ ]:
t0 = time.time()
assets = C.list_assets()
R = C.analyse_all(assets, workers=3)
print(f"{len(R)} recordings analysed in {time.time() - t0:.0f} s; peak memory of a worker "
      f"{resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.1f} GB")

## Data check

In [ ]:
meta = pd.DataFrame([dict(dish=r["dish"], date=r["date"], n_electrodes=r["n_el"],
                          n_spikes=r["meta"]["n_spikes_total"], unmapped=r["meta"]["n_spikes_unmapped"],
                          duration_s=r["meta"]["duration_s"], t_max=r["meta"]["t_max"], fs=r["meta"]["fs"],
                          mean_rate=r["mean_rate"], n_above_003=r["n_above"],
                          frames_match_times=r["meta"]["frames_match_times"],
                          sorted=r["meta"]["times_sorted"], grid_ok=r["meta"]["grid_ok"],
                          age=r["meta"]["subject_age"]) for r in R])
meta["day"] = C.day_index(R)
print("recordings per dish and date span:")
print(meta.groupby("dish").agg(n=("date", "size"), first=("date", "min"), last=("date", "max"),
                               span_days=("day", "max")).to_string())
print("\nelectrodes per recording:", meta.n_electrodes.min(), "to", meta.n_electrodes.max(),
      "(the paper: up to 1,024)")
print("recording duration (s):", meta.duration_s.min(), "to", meta.duration_s.max(),
      "(the paper: 30 minutes); last spike at", meta.t_max.min().round(1), "to", meta.t_max.max().round(1),
      "s after the first")
print("sampling rate (Hz):", sorted(meta.fs.unique()))
print("spike times equal (frame - first frame) / 20 kHz in every file:", meta.frames_match_times.all())
print("spike times sorted in every file:", meta.sorted.all())
print("electrode x, y equal the MaxOne grid (17.5 um pitch, 220 columns) of the electrode id:",
      meta.grid_ok.all())
print("spikes on channels with no electrode (dropped):", meta.unmapped.sum(), "of", meta.n_spikes.sum(),
      f"({meta.unmapped.sum() / meta.n_spikes.sum():.2e})")
print("subject age field:", meta.age.unique(), "(the files say it is the donor's E18, not DIV)")
import re
for field in ("experiment_description", "subject_description", "session_description"):
    texts = {re.sub(r"\d{5,6}", "<id>", r["meta"][field]) for r in R}
    print(f"{field} (dish ids and dates replaced by <id>), {len(texts)} distinct text(s):")
    for s in sorted(texts):
        print("   ", s)
print("first and last recording of each dish, age field:",
      meta.sort_values("day").groupby("dish").age.agg(["first", "last"]).drop_duplicates().to_dict("records"))
print("mean rate per electrode (Hz): %.4f to %.3f" % (meta.mean_rate.min(), meta.mean_rate.max()))
print("recordings with mean rate <= 0.03 Hz:", (meta.mean_rate <= C.RATE_HZ).sum(), "of", len(meta))
print("recordings with fewer than 10 electrodes above 0.03 Hz:", (meta.n_above_003 < C.MIN_POINTS).sum())
print(meta[["dish", "date", "day", "n_electrodes", "n_spikes", "mean_rate", "n_above_003"]].to_string())

Recordings deposited twice: no two files share a spike count, and recordings of different dishes
made at the same time (dishes 23280 and 23311 on 2024-05-16 to 06-02, 23282 and 23302 on
2024-06-21 to 07-01, which were recorded within a minute of each other) are compared below by the
correlation of their population spike counts in 100 ms bins. The time origin of each file is its
first spike, so a lag search of up to 60 s is made.

In [ ]:
print("files with the same spike count:", meta.n_spikes.duplicated().sum())
by = {(r["dish"], r["date"]): r for r in R}
pairs = [(a, b) for a, b in (("23280", "23311"), ("23282", "23302"), ("23302", "23317"))]
rows = []
for a, b in pairs:
    for (dish, date), r in by.items():
        if dish == a and (b, date) in by:
            p, q = r["pop"].astype(float), by[(b, date)]["pop"].astype(float)
            best = 0.0
            for lag in range(-600, 601, 1):
                if lag >= 0:
                    x, y = p[lag:], q[:len(q) - lag]
                else:
                    x, y = p[:lag], q[-lag:]
                if x.std() > 0 and y.std() > 0:
                    best = max(best, np.corrcoef(x, y)[0, 1])
            rows.append(dict(dishes=f"{a}/{b}", date=date, max_corr=round(best, 3)))
dup = pd.DataFrame(rows)
print(dup.to_string())
print("largest correlation at any lag:", dup.max_corr.max())

**Clock.** Each file holds one data stream, the spike detections, and no trial or annotation
table, so there is nothing to align; spike times run from the first detected spike, as the files
state (the recording-start frame is not stored). Electrode positions are static.

**Days in vitro.** The files give recording dates only. The dates of a dish span 10 to 32 days,
with some days missing. The Results say "Spontaneous activity became detectable approximately
2 – 3 days in vitro (DIV)" and that activity "was measured once a day". Four dishes are already
active (mean rate above 0.03 Hz) on their first recording, and three are near silent on it and
active one or two days later (23317 on day 1, 19880 and 23302 on day 2). So the first recording
falls at or just before the onset of activity, which the paper puts at 2 to 3 DIV. Assumption: DIV = day + d0 with d0 = 2
for every dish. Shown too: d0 = 1, 3, 4 and 5 for every dish (a dish active on its first recording
may have been first recorded after its onset), and two per-dish readings, O2 and O3, that put each
dish's first recording with mean rate above 0.03 Hz at DIV 2 or 3 (for the four dishes active on
day 0 these equal d0 = 2 and 3; 19880 and 23302 shift by 2 days and 23317 by 1). Claims whose result is
itself a DIV cannot rest on this and are blocked.

In [ ]:
D0 = 2
# Uniform d0 = 1 to 5, and two per-dish readings that put each dish's first recording with mean
# rate above 0.03 Hz at DIV 2 (O2) or 3 (O3), the onset the Results give.
D0_READINGS = (1, 2, 3, 4, 5, "O2", "O3")
onset_day = meta[meta.mean_rate > C.RATE_HZ].groupby("dish").day.min()
print("first day with mean rate above 0.03 Hz:", onset_day.to_dict())
T = C.fit_table(R)
T["active_S2"] = T.mean_rate > C.RATE_HZ
print("rows (recording x selection x fit):", len(T))
print("recordings with a spectrum under S1:", T[(T.sel == "S1") & (T.F == "F1")].b.notna().sum(),
      "and under S2:", T[(T.sel == "S2") & (T.F == "F1")].b.notna().sum(), "of", len(R))
print("electrodes kept under S1 per recording: median", int(T[T.sel == "S1"].n_kept.median()),
      "range", T[T.sel == "S1"].n_kept.min(), "to", T[T.sel == "S1"].n_kept.max())
print("\nPCA exponential decay rate b, by selection and fit reading (active recordings):")
print(T.dropna(subset=["b"]).groupby(["sel", "F"]).b.describe(percentiles=[.05, .5, .95]).round(4).to_string())
print("\nrecordings with PCA b >= 0.1 (the paper's Exponential group):")
print(T[T.b >= C.B_THRESHOLD][["dish", "day", "sel", "F", "mean_rate", "n_kept", "b", "c"]].round(4).to_string())


def d0_of(dish, d0=D0):
    """The offset DIV - day for a dish under reading d0."""
    return int(d0[1:]) - int(onset_day[dish]) if isinstance(d0, str) else d0


def div(df, d0=D0):
    """DIV of each row of df (columns dish and day) under reading d0."""
    return df.day + df.dish.map(lambda k: d0_of(k, d0))


def moving_average(df, col, d0=D0):
    """Centred 3-day moving average over DIV, pooled over dishes (DIV x-1 to x+1)."""
    dv, v = div(df, d0), df[col]
    xs = np.sort(dv.unique())
    return pd.Series([v[(dv >= x - 1) & (dv <= x + 1)].mean() for x in xs], index=xs)


def stays_below_from(days, vals, thr=C.B_THRESHOLD):
    """First day from which vals stay below thr; inf when the last value is at or above it."""
    above = np.flatnonzero(np.asarray(vals) >= thr)
    if not len(above):
        return days[0]
    return days[above[-1] + 1] if above[-1] + 1 < len(days) else np.inf


print("\nshare of fitted b at the bound 0 (below 1e-8), by selection and fit reading:")
bcols = ["b", "b_shuf", "avA1_b", "avA2_b"]
print(T.groupby(["sel", "F"])[bcols].agg(lambda v: (v.dropna() < 1e-8).mean()).round(3).to_string())
ratios = []
for r, d in zip(R, C.day_index(R)):
    ev = r["S1"].get("ev")
    if ev is not None and len(ev) > 100:
        ratios.append(dict(dish=r["dish"], day=d, n_pc=len(ev), l1_l10=ev[0] / ev[9], l1_l100=ev[0] / ev[99]))
ratios = pd.DataFrame(ratios)
ratios["rank"] = ratios.groupby("dish").day.rank(method="dense")
ratios["phase"] = np.where(ratios["rank"] <= 3, "first 3 recordings", "later")
print("\nS1, eigenvalue ratios (recordings with more than 100 PCs):")
print(ratios.groupby("phase")[["l1_l10", "l1_l100"]].describe(percentiles=[.5]).round(1).to_string())
print("for b >= 0.1 over ranks 1 to 100 with c >= 0, l1/l100 would have to exceed exp(9.9) =", round(np.exp(9.9)))


def by_reading(df, extra=()):
    return df.groupby(["sel", "F", *extra])

The values of b (rank as x) are far below 0.1 in nearly every recording. Under F1 two recordings
reach 0.1, both near silent (11 and 16 electrodes kept). Under F2 the median is 0 (76 to 79% of the
values at the bound) and six recordings reach 0.1 under S1: four near silent and two active, dish
23280 on day 2 (mean rate 0.72 Hz, b 0.464) and 23282 on day 7 (0.42 Hz, b 0.318). This is the scale problem said in the header. With x the rank, a decay rate b of 0.1
means the spectrum falls by a factor e over 10 ranks on top of the power law, so b of at least 0.1
over the first 100 ranks needs lambda_1/lambda_100 above e^9.9 (about 20,000); the ratios printed
above are far smaller. It is shown per claim below.

## dishes-analysed

> Seven dishes met the paper's inclusion rule (spontaneous activity observed continuously for more
> than 10 days) and were summarized.

Criterion: count the dishes in which recordings with activity above 0.03 Hz (S1, at least one
electrode above 0.03 Hz; S2, mean rate above 0.03 Hz) run continuously, without a day missing
activity, over more than 10 days. Whether a day without a recording breaks the run is not said;
both are shown: (i) a missing date breaks it, (ii) only a recorded day without activity breaks it.

In [ ]:
def longest_run(days, active, missing_breaks):
    best, start, prev = 0, None, None
    for d, a in zip(days, active):
        if not a:
            start, prev = None, None
            continue
        if start is None or (missing_breaks and d != prev + 1):
            start = d
        prev = d
        best = max(best, d - start + 1)
    return best


rows = []
for dish, g in meta.sort_values(["dish", "day"]).groupby("dish"):
    for sel, act in (("S1", g.n_above_003 > 0), ("S2", g.mean_rate > C.RATE_HZ)):
        for rule, mb in (("i, missing date breaks", True), ("ii, only inactive day breaks", False)):
            rows.append(dict(dish=dish, sel=sel, rule=rule, longest_run_days=longest_run(g.day.values, act.values, mb)))
runs = pd.DataFrame(rows)
print(runs.pivot_table(index="dish", columns=["sel", "rule"], values="longest_run_days").to_string())
counts = runs.assign(ok=runs.longest_run_days > 10).groupby(["sel", "rule"]).ok.sum()
print("\ndishes with a run of more than 10 days:")
print(counts.to_string())
print("dishes in the dataset:", meta.dish.nunique())

**Result: reproduced, resting on an assumption.** Under S1, with only a recorded day without
activity breaking a run (reading ii, assumed: a day with no recording is not a day without
activity), all 7 dishes have a run of more than 10 days (11 to 33 days). If a day with no
recording also breaks the run (reading i), 6 dishes do (23317 has 10 days); under S2, 2 or 4 do.
The dataset holds exactly 7 dishes. Under S1 nearly every recording counts as active (at least
one electrode above 0.03 Hz), so under S1 the count mostly measures how many days were recorded;
the result is fragile to both choices.

## activity-onset (blocked)

> Spontaneous activity became detectable at about 2 to 3 DIV.

Blocked, `metadata_insufficient`: the files give no DIV and no plating date. Shown for the record:
the first day (since the dish's first recording) with activity above 0.03 Hz.

In [ ]:
onset_rows = []
for dish, g in meta.sort_values("day").groupby("dish"):
    s1 = g[g.n_above_003 > 0].day.min()
    s2 = g[g.mean_rate > C.RATE_HZ].day.min()
    onset_rows.append(dict(dish=dish, first_day_S1=s1, first_day_S2=s2,
                             rate_first_recording=round(g.mean_rate.iloc[0], 4)))
print(pd.DataFrame(onset_rows).to_string())

## fig1b-async-to-bursts

> As the cultures matured, spiking changed from asynchronous firing to synchronous bursting.

Criterion: (a) mean pairwise Pearson correlation of the 100 ms counts, (b) the largest fraction
of electrodes active within one 100 ms bin; over all recordings against DIV, Spearman positive with
p < 0.05, and the dish median at 15 DIV and later above that at 3 to 5 DIV. Readings S1 and S2;
DIV = day + d0 (assumption, d0 = 2; d0 = 1, 3, 4, 5 and the per-dish readings O2 and O3
shown). The Spearman correlation is taken against DIV, which under O2 and O3 differs from the day
by dish. "Dish median" is read as the median over dishes of each
dish's median in the window; the median over recordings is shown too.

In [ ]:
def window_compare(df, col, d0, early=(3, 5), late=15):
    dv = div(df, d0)
    e = df[(dv >= early[0]) & (dv <= early[1])]
    l = df[dv >= late]
    de = e.groupby("dish")[col].median()
    dl = l.groupby("dish")[col].median()
    return dict(early_dish_median=de.median(), late_dish_median=dl.median(),
                early_rec_median=e[col].median(), late_rec_median=l[col].median(),
                n_early=len(e), n_late=len(l), dishes_early=len(de), dishes_late=len(dl))


rows = []
F1 = T[(T.F == "F1")].dropna(subset=["b"])
for sel, g in F1.groupby("sel"):
    for col in ("sync_corr", "sync_frac"):
        for d0 in D0_READINGS:
            rho, p = stats.spearmanr(div(g, d0), g[col])
            w = window_compare(g, col, d0)
            rows.append(dict(sel=sel, measure=col, d0=d0, spearman=rho, p=p, **w,
                             holds=(rho > 0) and (p < 0.05) and (w["late_dish_median"] > w["early_dish_median"])))
fig1b = pd.DataFrame(rows)
print(fig1b.round(4).to_string())
print("\nreadings of d0 under which both measures hold, by selection:")
print(fig1b.groupby(["sel", "d0"], sort=False).holds.all().unstack("d0").to_string())
print("\nper dish Spearman of synchrony with day (S1):")
print(F1[F1.sel == "S1"].groupby("dish").apply(
    lambda g: pd.Series({m: stats.spearmanr(g.day, g[m])[0] for m in ("sync_corr", "sync_frac")})).round(3).to_string())
print("\nrange of sync_corr", F1.sync_corr.min().round(4), F1.sync_corr.max().round(4),
      "; sync_frac", F1.sync_frac.min().round(4), F1.sync_frac.max().round(4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for dish, g in F1[F1.sel == "S1"].groupby("dish"):
    ax[0].plot(div(g), g.sync_corr, "o-", ms=3, label=dish)
    ax[1].plot(div(g), g.sync_frac, "o-", ms=3)
ax[0].set(xlabel="DIV (day + 2, assumed)", ylabel="mean pairwise correlation")
ax[1].set(xlabel="DIV (day + 2, assumed)", ylabel="largest fraction active in a bin")
ax[0].legend(fontsize=7)
plt.tight_layout()

**Result: reproduced, resting on the assumption on d0.** Over all recordings the mean pairwise
correlation rises with day (Spearman 0.487 under S1, 0.384 under S2, p < 0.0001), and so does the
largest fraction of electrodes active in one bin (0.301, p 0.0007; 0.253, p 0.0072). Under O2 and O3, against each dish's DIV, the correlations
are 0.462 and 0.269 (S1) and 0.373 and 0.232 (S2), p at most 0.014. The dish median
from 15 DIV exceeds that at 3 to 5 DIV for both measures and both selections under d0 = 2, 3, 4, 5
and the per-dish readings O2 and O3 (correlation 0.330 against 0.063 under S1, d0 = 2). Only d0 = 1,
which would put activity at 1 DIV, before the onset the paper gives, fails: for the fraction active
under S1 (0.816 late against 0.892 early) and for both measures under S2. Not every dish shows the
rise: the correlation falls with day in 23282 (Spearman -0.227) and the fraction active in 23280
(-0.156) and 23282 (-0.273).

## fig1d-power-law-emerges (blocked)

> In early cultures the variance spectrum decays faster than a power function; as the culture
> matures it follows a power function, linear on log-log axes.

Criterion: in a majority of the seven dishes, the recordings of the first three recorded DIV have
b >= 0.1 and those from 15 DIV on have b < 0.1 (F1 and F2). Computed as written, and with the
threshold-free trend of b and c against day.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for d0 in D0_READINGS:
        ok = 0
        for dish, h in g.groupby("dish"):
            first3 = h[h.day.isin(sorted(h.day.unique())[:3])]
            late = h[div(h, d0) >= 15]
            ok += bool((first3.b >= 0.1).all() and len(late) and (late.b < 0.1).all())
        rows.append(dict(sel=sel, F=F, d0=d0, dishes_meeting=ok,
                         first3_b_ge_01=(g.groupby("dish").apply(lambda h: (h[h.day.isin(sorted(h.day.unique())[:3])].b >= 0.1).sum())).sum()))
print(pd.DataFrame(rows).to_string())
print("\nb and c in the first three recordings and from 15 DIV on (d0 = 2), median over recordings:")
g = T.dropna(subset=["b"]).copy()
g["rank"] = g.groupby(["dish", "sel", "F"]).day.rank(method="dense")
g["phase"] = np.where(g["rank"] <= 3, "first3", np.where(div(g) >= 15, "from15", "between"))
print(g.groupby(["sel", "F", "phase"])[["b", "c"]].median().round(4).to_string())
for (sel, F), h in by_reading(T.dropna(subset=["b"])):
    print(sel, F, "Spearman of b with day: rho %.3f p %.2g; of c with day: rho %.3f p %.2g" % (
        *stats.spearmanr(h.day, h.b), *stats.spearmanr(h.day, h.c)))

In [ ]:
# Variants of the fit range and of x, shown for the question to the authors (not part of the criterion).
rows = []
for r, d in zip(R, C.day_index(R)):
    ev = r["S1"].get("ev")
    if ev is None:
        continue
    n = len(ev)
    rows.append(dict(dish=r["dish"], day=d, first3=d <= sorted({dd for rr, dd in zip(R, C.day_index(R)) if rr["dish"] == r["dish"]})[2],
                     b_all=C.fit_decay(ev, "F1")[1], b_top100=C.fit_decay(ev[:100], "F1")[1],
                     b_top30=C.fit_decay(ev[:30], "F1")[1],
                     b_x_over_n=C.fit_decay(ev, "F1", np.arange(1, n + 1) / n)[1],
                     # The Figure 1D legend puts the rank on the vertical axis and the variance on the
                     # horizontal one; these fit rank as the function of variance (raw, and as a share).
                     b_rank_of_var=C.fit_decay(np.arange(1, n + 1.0), "F1", ev)[1],
                     b_rank_of_share=C.fit_decay(np.arange(1, n + 1.0), "F1", ev / ev.sum())[1]))
var = pd.DataFrame(rows)
print("S1, F1: median b in the first three recordings of each dish and in the others, and the share >= 0.1")
vcols = ["b_all", "b_top100", "b_top30", "b_x_over_n", "b_rank_of_var", "b_rank_of_share"]
print(var.groupby("first3")[vcols].agg(["median", lambda v: (v >= 0.1).mean()]).round(4).T.to_string())
print("Spearman of each variant with day:")
for col in vcols:
    print("  %-16s rho %.3f p %.2g" % (col, *stats.spearmanr(var.day, var[col], nan_policy="omit")))
# The fig1d criterion with b from rank fitted as a function of the variance (variant, not the claims file's x).
rows = []
for r, d in zip(R, C.day_index(R)):
    for sel in ("S1", "S2"):
        ev = r[sel].get("ev")
        if ev is None:
            continue
        for F in ("F1", "F2"):
            rows.append(dict(dish=r["dish"], day=d, sel=sel, F=F,
                             b=C.fit_decay(np.arange(1, len(ev) + 1.0), F, ev)[1]))
rv = pd.DataFrame(rows)
crit = []
for (sel, F), g in rv.groupby(["sel", "F"]):
    for d0 in D0_READINGS:
        ok = 0
        for dish, h in g.groupby("dish"):
            first3 = h[h.day.isin(sorted(h.day.unique())[:3])]
            late = h[div(h, d0) >= 15]
            ok += bool((first3.b >= 0.1).all() and len(late) and (late.b < 0.1).all())
        crit.append(dict(sel=sel, F=F, d0=d0, dishes_meeting=ok))
    first3 = g.groupby("dish", group_keys=False).apply(lambda h: h[h.day.isin(sorted(h.day.unique())[:3])])
    print(sel, F, "rank-of-variance b: share >= 0.1 in the first three recordings %.2f, from 15 DIV (d0 = 2) %.2f" % (
        (first3.b >= 0.1).mean(), (g[div(g) >= 15].b >= 0.1).mean()))
print(pd.DataFrame(crit).pivot_table(index=["sel", "F"], columns="d0", values="dishes_meeting", sort=False).to_string())


In [ ]:
# Null for the scale of b: independent Poisson counts with each kept electrode's rate (S1), no
# correlations at all. Run on the first and last S1 recording of each dish.
rng_null = np.random.default_rng(20201)
rows = []
days_all = C.day_index(R)
for dish in sorted({r["dish"] for r in R}):
    recs = [(d, r) for d, r in zip(days_all, R) if r["dish"] == dish and "ev" in r["S1"]]
    for d, r in (recs[0], recs[-1]):
        lam = r["rates"][r["rates"] > C.RATE_HZ] * C.BIN_S
        Xp = rng_null.poisson(lam[:, None], size=(len(lam), C.N_BINS)).astype(float)
        evp = C.pca(Xp)[0]
        rows.append(dict(dish=dish, day=d, n_el=len(lam), b_data_F1=C.fit_decay(r["S1"]["ev"], "F1")[1],
                         b_poisson_F1=C.fit_decay(evp, "F1")[1], b_data_F2=C.fit_decay(r["S1"]["ev"], "F2")[1],
                         b_poisson_F2=C.fit_decay(evp, "F2")[1], c_data_F1=C.fit_decay(r["S1"]["ev"], "F1")[2],
                         c_poisson_F1=C.fit_decay(evp, "F1")[2]))
        del Xp
nullb = pd.DataFrame(rows)
print(nullb.round(4).to_string())
print("median b, data against the Poisson null:", nullb[["b_data_F1", "b_poisson_F1", "b_data_F2", "b_poisson_F2"]].median().round(4).to_dict())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ex = [r for r in R if r["dish"] == "19880"]
days_ex = [d for r, d in zip(R, C.day_index(R)) if r["dish"] == "19880"]
cm = plt.cm.viridis
for r, d in zip(ex, days_ex):
    ev = r["S1"].get("ev")
    if ev is not None:
        ax[0].loglog(np.arange(1, len(ev) + 1), ev / ev[0], color=cm(d / max(days_ex)), lw=0.8)
        ax[1].loglog(np.arange(1, len(ev) + 1), r["S1"]["ev_shuf"] / r["S1"]["ev_shuf"][0],
                     color=cm(d / max(days_ex)), lw=0.8)
ax[0].set(title="dish 19880, S1, colour = day", xlabel="PC rank", ylabel="variance / first")
ax[1].set(title="same, shuffled", xlabel="PC rank")
for (sel, F), h in by_reading(T[(T.sel == "S1")].dropna(subset=["b"])):
    ax[2].semilogy(div(h), h.b.clip(lower=1e-5), "o", ms=3, label=F, alpha=0.6)
ax[2].axhline(0.1, color="k", ls="--")
ax[2].set(xlabel="DIV (day + 2, assumed)", ylabel="b (clipped at 1e-5)", title="PCA b, S1")
ax[2].legend()
plt.tight_layout()
print("b values below 1e-5 (clipped in the plot):", (T.b < 1e-5).sum(), "of", T.b.notna().sum())

**Result: blocked, `methods_underspecified`.** As written, no dish meets the criterion under any
fit, selection or d0 reading. The b fitted here cannot reach the paper's scale: with x the PC rank,
the median b is 0.0055 (S1, F1), and b of at least 0.1 over the first 100 ranks needs
lambda_1/lambda_100 above about 19,930, while the largest ratio is 5,503 (median 70.9 in the first
three recordings, 844.5 later). Independent Poisson counts at the recorded rates, with no
correlation at all, give the same b (median 0.0114 against 0.0131 in the data, F1, 14 recordings).
The variants of the fit range (top 30 or 100 ranks) and x over n do not separate early from late
recordings either. Fitting the rank as a function of the variance, the axes the Figure 1D legend
describes, gives b at or above 0.1 in 0.86 of the first three recordings and 0.20 from 15 DIV
(S1, F1), but meets the criterion in 1 dish (S1, F1), 2 to 4 (S2, F1) and none under F2. The fitted
quantity is asked of a reader in `questions/002020.yaml`. Without the threshold, b falls with day
only under S1 F1 (Spearman -0.256, p 0.0044).

## fig1e-example-dish, fig1f-transition-15div, fig1f-exponent-1p2 (blocked)

Blocked: the shift DIV and the transition DIV need days in vitro, which the files do not give, and
the 0.1 boundary on b is met from the first active recording in every dish (above). The exponent
claim depends on the transition. Shown for the record: per dish, the first day from which b stays
below 0.1, and the median c over recordings from that day (F1, F2).

In [ ]:
# Readings of "established": (a) the pooled 3-day moving average over DIV of b, the first DIV from
# which it stays below 0.1; (b) per dish, the first day from which b stays below 0.1 (inf when the
# last recording is at or above 0.1, so that such a dish counts as later than every other), median
# over dishes.
rows, dish_rows = [], []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    per = {}
    for dish, h in g.groupby("dish"):
        h = h.sort_values("day")
        per[dish] = stays_below_from(h.day.values, h.b.values)
    for d0 in D0_READINGS:
        rho, p = stats.spearmanr(div(g, d0), g.b)
        ma = moving_average(g, "b", d0)
        div_a = stays_below_from(ma.index.values, ma.values)
        divs_b = [v + d0_of(k, d0) for k, v in per.items()]
        div_b = float(np.median(divs_b))
        after_a = g[div(g, d0) >= div_a]
        after_b = pd.concat([g[(g.dish == k) & (g.day >= v)] for k, v in per.items()])
        cma = moving_average(g, "c", d0)
        rows.append(dict(sel=sel, F=F, d0=d0, spearman_b_day=rho, p=p, div_a=div_a, div_b=div_b,
                         dishes_never=int(np.isinf(list(per.values())).sum()),
                         c_median_after_a=after_a.c.median(), c_median_after_b=after_b.c.median(),
                         c_ma_last_div=cma.iloc[-1],
                         transition_holds=(rho < 0) and (p < 0.05) and (abs(div_a - 15) <= 1.5 or abs(div_b - 15) <= 1.5)))
    for k, v in per.items():
        dish_rows.append(dict(sel=sel, F=F, dish=k, first_day_b_stays_below=v))
tr = pd.DataFrame(rows)
print(tr.to_string(float_format=lambda v: f"{v:.3g}"))
print("\nper dish, first day from which b stays below 0.1:")
print(pd.DataFrame(dish_rows).set_index(["dish", "sel", "F"]).first_day_b_stays_below.unstack(["sel", "F"]).to_string())

In [ ]:
# fig1e-example-dish, per dish: the shift (first DIV from which b stays below 0.1, as above), and the
# exponent as (a) the median c from the shift to 19 DIV, (b) c of the recording nearest 19 DIV,
# (c) a pure power law fitted to the dish's spectra from the shift to 19 DIV, pooled (each spectrum
# divided by its sum, a standard choice so that spectra of different size share one scale).
rows = []
by_rec = {(r["dish"], d): r for r, d in zip(R, C.day_index(R))}
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for d0 in D0_READINGS:
        for dish, h in g.groupby("dish"):
            h = h.sort_values("day")
            start = stays_below_from(h.day.values, h.b.values)
            dv = div(h, d0)
            if np.isinf(start):
                rows.append(dict(sel=sel, F=F, d0=d0, dish=dish, shift_div=np.inf))
                continue
            seg = h[(h.day >= start) & (dv <= 19)]
            near19 = h.iloc[[int(np.argmin(np.abs(dv.values - 19)))]]
            xs, ys = [], []
            for day in seg.day:
                ev = by_rec[(dish, day)][sel]["ev"]
                xs.append(np.arange(1, len(ev) + 1.0))
                ys.append(ev / ev.sum())
            c_pool = C.fit_power(np.concatenate(ys), np.concatenate(xs)) if xs else np.nan
            shift_div = start + d0_of(dish, d0)
            ex = dict(a=seg.c.median(), b=near19.c.iloc[0], c=c_pool)
            rows.append(dict(sel=sel, F=F, d0=d0, dish=dish, shift_div=shift_div, div_near19=div(near19, d0).iloc[0],
                             c_a=ex["a"], c_b=ex["b"], c_c=ex["c"],
                             holds=abs(shift_div - 10) <= 1 and (seg.b < 0.1).all()
                             and any(abs(v - 1.30) <= 0.13 for v in ex.values())))
fe = pd.DataFrame(rows)
print("S1, d0 = 2, per dish:")
print(fe[(fe.sel == "S1") & (fe.d0 == 2)].to_string(float_format=lambda v: f"{v:.3g}"))
print("\ndishes meeting the fig1e criterion, by reading:")
print(fe.groupby(["sel", "F", "d0"], sort=False).holds.sum().unstack("d0").to_string())
print("dishes with an exponent within 1.30 +- 0.13 under some reading, whatever the shift:")
fe["exp_ok"] = fe[["c_a", "c_b", "c_c"]].sub(1.30).abs().le(0.13).any(axis=1)
print(fe[fe.d0 == 2].groupby(["sel", "F"]).exp_ok.sum().to_string())

**Result: blocked, `methods_underspecified`, and the files give no DIV.** b falls with day only
under S1 F1 (Spearman -0.256, p 0.0045, or -0.218, p 0.016, against the DIV of O2 and O3; -0.116 to -0.029
otherwise, p at least 0.22). b
stays below 0.1 from the first or nearly the first recording: the DIV of "established" is 0 to 5
under reading (a) and 1 to 8 under reading (b), over every fit, selection and d0, never within
15 plus or minus 1.5, which reflects the scale of b above, not a transition. The median c after
it is 0.72 to 0.81 (F1) and 1.98 to 2.24 (F2), and the moving average of c at the last DIV 0.58 (F1)
and 3.72 (F2); none is within 1.2 plus or minus 0.12 (fig1f-exponent-1p2, blocked with its
dependency). For fig1e-example-dish no dish meets the criterion under any reading: the shift falls
at DIV 2 to 5 in most dishes under d0 = 2, and the one dish with a shift at DIV 10 (23302, S1, F2)
has exponents 4.41, 0.39 and 1.66 under readings (a), (b) and (c).

## fig1d-shuffle (blocked)

> Temporal shuffling leaves the variance spectrum unchanged early in development but, once the
> power law has emerged, disrupts it and returns it to exponential decay.

Criterion: early recordings are the Exponential group (b >= 0.1), late ones the Power-law group;
(i) in early recordings the median ratio of shuffled to original b within 0.9 to 1.1; (ii) in late
recordings shuffled b >= 0.1 in a majority and above the original b, one-sided Wilcoxon p < 0.05.
As written, and with early and late taken as 3 to 5 DIV and from 15 DIV (d0 = 2), a variant.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b", "b_shuf"])):
    for name, early, late in (("groups by b", g[g.b >= 0.1], g[g.b < 0.1]),
                              ("DIV 3-5 / 15+ (d0=2)", g[div(g).between(3, 5)], g[div(g) >= 15])):
        ratio = (early.b_shuf / early.b).median() if len(early) else np.nan
        p = stats.wilcoxon(late.b_shuf, late.b, alternative="greater").pvalue if len(late) > 1 else np.nan
        rows.append(dict(sel=sel, F=F, split=name, n_early=len(early), early_median_ratio=ratio,
                         n_late=len(late), late_frac_shuf_ge_01=(late.b_shuf >= 0.1).mean(),
                         late_median_b=late.b.median(), late_median_b_shuf=late.b_shuf.median(),
                         late_median_c=late.c.median(), late_median_c_shuf=late.c_shuf.median(), wilcoxon_p=p))
print(pd.DataFrame(rows).round(4).to_string())

**Result: blocked, `methods_underspecified`.** The early group of the criterion (PCA b at or above
0.1) holds 2 recordings (S1, F1), 6 (S1, F2), 0 (S2, F1) and 2 (S2, F2), most of them near silent,
so part (i) cannot be judged. Part (ii) as written fails: the shuffled b is at or above 0.1 in 0 to
2.6% of late recordings (0 to 4.5% in the DIV variant). Whether the shuffle raises b depends on the reading: under S1, F1 it does
not (median 0.0054 against 0.0043 shuffled, Wilcoxon p 0.98), while under S1, F2, S2, F1 and S2, F2
the shuffled b is higher (one-sided Wilcoxon p at most 0.0004; medians 0 against 0.0051, 0.0039
against 0.0044 and 0 against 0.0047, groups by b), but it stays far below 0.1. The shuffle
flattens the late spectra (median c 0.75 to 0.25, S1, F1). With early and late taken as 3 to 5 DIV
and from 15 DIV (d0 = 2), a variant, the median ratio of shuffled to original b is 0.72 to 1.07
early. As uncorrelated Poisson counts give the same b as the data, a shuffled b at or above 0.1
cannot be expected whatever the data; whether the flattening is the paper's reverting to
exponential decay depends on how its b is computed.


## fig2c-locality-spectrum (blocked)

> The locality spectrum (the number of electrodes with local Moran's I above threshold, against PC
> index) is flat early in development and becomes steeper as the culture matures.

Criterion: under M1 and M2 and L1 and L2, the Spearman correlation of the locality slope with DIV,
over all recordings, negative with p < 0.05, and the median slope at 3 to 5 DIV closer to zero than
that at 15 DIV and later. Slopes are from the original PCs; w_ii = 0 (w_ii = 1 shown as a variant).
Readings S1 and S2; d0 = 1 to 5, O2 and O3 for the windows and for the Spearman correlation against
DIV (the same for every uniform d0, not under O2 and O3).

In [ ]:
rows = []
L = T[T.F == "F1"].dropna(subset=["b"])
for sel, g in L.groupby("sel"):
    g = g.copy()
    for key in ("M1", "M2"):
        # Locality as a share of the kept electrodes: the L1 slope divided by the electrode count
        # (the L2 slope does not change), a variant for the count's growth under S1.
        g[f"loc_{key}/n_L1"] = g[f"loc_{key}_L1"] / g.n_kept
    for key in ("M1", "M2", "M1_self", "M2_self", "M1/n", "M2/n"):
        for Lr in (("L1", "L2") if "/n" not in key else ("L1",)):
            col = f"loc_{key}_{Lr}"
            h = g.dropna(subset=[col])
            for d0 in D0_READINGS:
                dv = div(h, d0)
                rho, p = stats.spearmanr(dv, h[col])
                me, ml = h[dv.between(3, 5)][col].median(), h[dv >= 15][col].median()
                rows.append(dict(sel=sel, moran=key, slope=Lr, d0=d0, n=len(h), spearman=rho, p=p,
                                 median_3_5=me, median_15=ml,
                                 holds=(rho < 0) and (p < 0.05) and (abs(me) < abs(ml))))
fig2c = pd.DataFrame(rows)
print(fig2c.to_string(float_format=lambda v: f"{v:.4g}"))
main = fig2c[fig2c.moran.isin(["M1", "M2"])]
print("\nmain readings (w_ii = 0) holding:", main.holds.sum(), "of", len(main))
print(main.groupby(["sel", "moran", "slope"]).holds.sum().to_string())
print("\nmedian slope at 3 to 5 DIV against from 15 DIV, M2 L1, S1, by d0:")
print(fig2c[(fig2c.sel == "S1") & (fig2c.moran == "M2") & (fig2c.slope == "L1")][["d0", "median_3_5", "median_15"]].to_string(index=False))

Magnitude of local Moran's I under M1 and M2, on one recording (dish 23312, its last day), to see
what the threshold 0.0001 does under each reading.

In [ ]:
a = [x for x in assets if x["dish"] == "23312"][-1]
t_, e_, x_, y_, _ = C.load_recording(a["asset_id"])
X_ = C.bin_counts(t_, e_, len(x_))
rates_ = X_.sum(axis=1) / C.DURATION_S
keep_ = rates_ > C.RATE_HZ
ev_, V_, _ = C.pca(X_[keep_])
W_ = C.neighbour_matrix(x_[keep_], y_[keep_])
Z_ = V_ - V_.mean(axis=0)
I2 = Z_ * (W_ @ Z_)
I1 = I2 / ((Z_ ** 2).mean(axis=0))
for name, I in (("M1", I1), ("M2", I2)):
    print(f"{a['path']}: {name} median |I| {np.median(np.abs(I)):.3g}; share of values > 0 {np.mean(I > 0):.3f}; "
          f"share > 0.0001 {np.mean(I > C.MORAN_THRESHOLD):.3f}")
del X_, V_, Z_, I1, I2

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for r, d in zip(R, C.day_index(R)):
    if r["dish"] != "23312" or "ev" not in r["S1"]:
        continue
    loc = r["S1"]["loc_M2"]
    sm = np.convolve(loc, np.ones(10) / 10, mode="valid")
    ax[0].semilogx(np.arange(1, len(sm) + 1), sm, color=plt.cm.viridis(d / 25), lw=0.8)
ax[0].set(title="dish 23312, S1, M2, 10-PC moving average", xlabel="PC index", ylabel="locality")
h = L[L.sel == "S1"]
ax[1].plot(div(h), h["loc_M2_L1"], "o", ms=3)
ax[1].set(xlabel="DIV (day + 2, assumed)", ylabel="locality slope (M2, L1)")
plt.tight_layout()

**Result: blocked, `methods_underspecified`: the readings disagree.** Under M2 with L1 (w_ii = 0)
the slope falls with day (Spearman -0.318, p 0.00036, S1; -0.401, p 1.2e-05, S2; against the DIV of O2 and O3,
-0.299 and -0.385) and the median slope
at 3 to 5 DIV is closer to zero than from 15 DIV for d0 = 1 to 5 and O3, not O2. Under M1 and under
L2 the Spearman correlation is not negative with p < 0.05 (for example 0.009, p 0.92, S1, M1, L1), so
12 of the 56 main readings hold. The claims file says a claim is blocked where its readings give
different outcomes. Locality as a share of the kept electrodes (M2, L1) gives the same result as M2
(Spearman -0.285, p 0.0015, S1). Even under M2 with L1 the early spectra are not flat: with d0 = 2
the medians are -104.2 early and -105.7 late (S1), and only from d0 = 3 is the early median much
nearer zero (-23.95). Under M1 the threshold 0.0001 keeps nearly every positive value (share above it
0.394, share positive 0.409), while under M2 it keeps 0.057.

## fig2b-spatial-structure (blocked)

> Early in development (3 DIV) all PCs correspond to a few scattered electrodes; in mature cultures
> (15 DIV) dominant PCs correspond to localized clusters of electrodes, while minor PCs remain on
> globally distributed, sporadic electrodes.

Criterion: fig2c-locality-spectrum reproduced and, in a majority of dishes at 15 DIV (or the nearest
recorded DIV), the mean locality of the first 10 PCs above that of the last 10, while at 3 DIV it
does not exceed it by more than 10%. M1 and M2 (w_ii = 0); S1 and S2; d0 = 1 to 5, O2 and O3.

In [ ]:
days = C.day_index(R)
rows = []
for sel in ("S1", "S2"):
    for key in ("M1", "M2"):
        for d0, tie in [(d, "earlier") for d in D0_READINGS] + [(d, "later") for d in D0_READINGS]:
            late_ok = early_ok = n_dish = 0
            detail = []
            for dish in sorted({r["dish"] for r in R}):
                recs = [(d, r) for d, r in zip(days, R) if r["dish"] == dish and f"loc_{key}" in r[sel]]
                if not recs:
                    continue
                n_dish += 1
                pick = {}
                for target in (3, 15):
                    sign = 1 if tie == "earlier" else -1
                    d, r = min(recs, key=lambda dr: (abs(dr[0] + d0_of(dish, d0) - target), sign * dr[0]))
                    loc = r[sel][f"loc_{key}"]
                    pick[target] = (d + d0_of(dish, d0), loc[:10].mean(), loc[-10:].mean())
                late_ok += pick[15][1] > pick[15][2]
                early_ok += pick[3][1] <= 1.1 * pick[3][2]
                detail.append((dish, *[round(v, 1) for v in pick[3]], *[round(v, 1) for v in pick[15]]))
            rows.append(dict(sel=sel, moran=key, d0=d0, tie=tie, dishes=n_dish, late_first10_gt_last10=late_ok,
                             early_within_10pct=early_ok,
                             holds_own_part=(late_ok > n_dish / 2) and (early_ok > n_dish / 2)))
            if sel == "S1" and d0 == D0 and tie == "earlier":
                print(sel, key, "per dish (DIV, first 10, last 10) at about 3 DIV, then at about 15 DIV:")
                for row in detail:
                    print("  ", row)
fig2b = pd.DataFrame(rows)
print(fig2b.to_string())
print("\nreadings meeting the fig2b part of the criterion (without fig2c):")
print(fig2b[fig2b.holds_own_part][["sel", "moran", "d0", "tie"]].to_string(index=False))

**Result: blocked, `methods_underspecified`, as it depends on fig2c-locality-spectrum.** Its own part
holds only under S2 with M1 and d0 = 2 to 5 (late 4 to 6 of 7 dishes, early within 10% in 4 to 7 of
7), where fig2c-locality-spectrum fails. Under S1 (M1 and M2) and S2 with M2, at about 15 DIV the
first 10 PCs are more local than the last 10 in 7 of 7 dishes, but at about 3 DIV they are within
10% of the last 10 in only 0 to 2 of 7. Breaking ties toward the later recording changes no reading's
result.

## fig2d-locality-slope (blocked)

> The slope of the locality spectrum confirms that spatial structure emerges progressively and is
> absent in the initial stages of development.

Criterion: on the 3-day moving average of the slope over DIV (pooled over dishes; a centred
window of DIV d-1 to d+1, a standard reading of "3-day moving average"), near zero at the first
recorded DIV (magnitude below 10% of its most negative value) and falling over DIV, without
returning to zero once below half of its most negative value. M1, M2, L1, L2; S1 and S2. With one
d0 for all dishes the moving average is the same curve shifted, so d0 does not change the result;
the per-dish readings O2 and O3 are shown too.

In [ ]:
rows = []
mas = {}
for sel, g in L.groupby("sel"):
    for key, Lr, d0 in [(k, l, d) for k in ("M1", "M2") for l in ("L1", "L2") for d in (2, "O2", "O3")]:
            col = f"loc_{key}_{Lr}"
            ma = moving_average(g.dropna(subset=[col]), col, d0)
            mas[(sel, key, Lr, d0)] = ma
            mn = ma.min()
            first = ma.iloc[0]
            below_half = np.flatnonzero(ma.values <= 0.5 * mn)
            after = ma.values[below_half[0]:] if len(below_half) else np.array([])
            returns = bool(len(after) and (np.abs(after) < 0.1 * abs(mn)).any())
            rho, p = stats.spearmanr(ma.index, ma.values)
            rows.append(dict(sel=sel, moran=key, slope=Lr, d0=d0, first_ma=first, most_negative=mn,
                             first_over_min=first / mn if mn < 0 else np.nan,
                             div_most_negative=ma.idxmin(), ma_trend_rho=rho, ma_trend_p=p,
                             returns_to_zero=returns,
                             holds=(mn < 0) and abs(first) < 0.1 * abs(mn) and rho < 0 and not returns))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.4g}"))
print("\nreadings holding:", int(pd.DataFrame(rows).holds.sum()), "of", len(rows))
print("\nmoving average, S1, M2, L1, by DIV (d0 = 2):")
print(mas[("S1", "M2", "L1", 2)].round(1).to_string())
print("\nS1, M2, L1: the recordings in the first window of the moving average (DIV first to first + 1):")
for d0 in (2, "O2", "O3"):
    h = L[L.sel == "S1"].dropna(subset=["loc_M2_L1"]).assign(DIV=lambda q: div(q, d0))
    first = h[h.DIV <= h.DIV.min() + 1].sort_values("DIV")
    print(f"d0 = {d0}:")
    print(first[["dish", "day", "DIV", "n_kept", "mean_rate", "loc_M1_L1", "loc_M2_L1"]].round(4).to_string(index=False))
print("\nL2 readings, Spearman of the moving average with DIV:")
print(pd.DataFrame(rows).query("slope == 'L2'")[["sel", "moran", "d0", "ma_trend_rho", "ma_trend_p", "first_over_min", "returns_to_zero"]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))


**Result: blocked, `methods_underspecified`, as it depends on fig2c-locality-spectrum; its own
readings also disagree.** Under M2 with L1 and d0 = 2 the moving average falls from -34.5 at DIV 2 to
-186.1 at DIV 26 (Spearman with DIV -0.729, p 1.5e-06, S1; -0.766, S2) and does not return to zero,
but its first value is 19% of its most negative value (S1 and S2), above the 10% bound for absent.
4 of 24 readings hold, all under S1 with L1 (M1 and M2) and the per-dish readings O2 and O3. These
passes rest on the first window of the moving average, which under O2 and O3 holds only two
near-silent recordings from before their dish's onset (19880 day 0, 14 electrodes kept, 0.009 Hz;
23317 day 0, 11 electrodes, 0.007 Hz), with a mean slope of 0.36; under d0 = 2 the same window
holds ten recordings and the first value is -34.5. Under L2 no reading holds: under S1 the moving
average rises (Spearman 0.270 to 0.423) and returns to zero; under S2 it does not fall
significantly (Spearman -0.329 to 0.098, p at least 0.076) and its first value is 24 to 55% of its
most negative value.


## fig2d-shuffle-locality (blocked)

> Temporal shuffling, which removes inter-electrode correlations, disrupts the spatial structure of
> PCs; the locality spectrum becomes nearly independent of PC index.

Criterion: data with established structure read as (a) recordings from 15 DIV on (d0 = 1 to 5, O2 and O3),
(b) recordings in the PCA Power-law group (b < 0.1; under F1 and F2 this is nearly every active
recording). Under each of M1, M2, L1, L2, the median shuffled slope below 10% of the median
original slope in magnitude, and the shuffled slopes less negative than the original by a one-sided
Wilcoxon signed-rank test, p < 0.05.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    sets = [(f"a, DIV >= 15, d0={d0}", g[div(g, d0) >= 15]) for d0 in D0_READINGS]
    sets.append(("b, Power-law group", g[g.b < 0.1]))
    if F == "F2":
        sets = sets[-1:]
    for name, h in sets:
        for key in ("M1", "M2"):
            for Lr in ("L1", "L2"):
                o, s = h[f"loc_{key}_{Lr}"], h[f"locshuf_{key}_{Lr}"]
                ok = o.notna() & s.notna()
                o, s = o[ok], s[ok]
                p = stats.wilcoxon(s, o, alternative="greater").pvalue if len(o) > 1 else np.nan
                rows.append(dict(sel=sel, F=F, set=name, moran=key, slope=Lr, n=len(o),
                                 median_orig=o.median(), median_shuf=s.median(),
                                 ratio=abs(s.median()) / abs(o.median()), wilcoxon_p=p,
                                 holds=abs(s.median()) < 0.1 * abs(o.median()) and p < 0.05))
fig2ds = pd.DataFrame(rows)
print(fig2ds.to_string(float_format=lambda v: f"{v:.4g}"))
print("\nreadings holding:", fig2ds.holds.sum(), "of", len(fig2ds))
print(fig2ds.groupby(["set", "moran", "slope"]).holds.all().to_string())

**Result: blocked, `methods_underspecified`, as it depends on fig2c-locality-spectrum.** Shuffling
makes the slopes less negative under every reading and set (one-sided Wilcoxon p at most 2.4e-05).
Under M2 with L1 the shuffled median is below 10% of the original in every set, for S1 and S2
(ratio 0.001 to 0.094), but under M2 with L2 it is not in any set (0.12 to 0.73), so no set meets
the bound under each of M1, M2, L1 and L2, as the criterion requires (41 of 72 rows hold).

## fig3c-avalanches-emerge (not reproduced)

> Across seven dishes the cascade size distribution shifted from exponential to power-law decay at
> about 15 DIV, with a power exponent of about 1.5.

The shift DIV needs days in vitro, but the result does not depend on them (see below). Computed under
every reading of the criterion: sizes A1 and A2, fits F1 and F2, selections S1 and S2, the shift
(a) on the pooled 3-day moving average of the avalanche b over DIV and (b) per dish, the median
over dishes (a dish whose last recording has b at or above 0.1 counts as later than every other),
for each d0. The exponent is the median c over recordings after the shift.

In [ ]:
rows, dish_rows = [], []
for (sel, F), g0 in by_reading(T):
    for A in ("A1", "A2"):
        bcol, ccol = f"av{A}_b", f"av{A}_c"
        g = g0.dropna(subset=[bcol])
        per = {}
        for dish in sorted(T.dish.unique()):
            h = g[g.dish == dish].sort_values("day")
            per[dish] = stays_below_from(h.day.values, h[bcol].values) if len(h) else np.nan
            dish_rows.append(dict(sel=sel, F=F, A=A, dish=dish, n=len(h), first_day_stays_below=per[dish]))
        for d0 in D0_READINGS:
            ma = moving_average(g, bcol, d0)
            div_a = stays_below_from(ma.index.values, ma.values)
            divs_b = [v + d0_of(k, d0) for k, v in per.items() if not np.isnan(v)]
            div_b = float(np.median(divs_b))
            c_a = g[div(g, d0) >= div_a][ccol].median()
            c_b = pd.concat([g[(g.dish == k) & (g.day >= v)] for k, v in per.items() if not np.isnan(v)])[ccol].median()
            rows.append(dict(sel=sel, F=F, A=A, d0=d0, n=len(g), div_a=div_a, c_after_a=c_a, div_b=div_b, c_after_b=c_b,
                             dishes_never=int(np.isinf([v for v in per.values() if not np.isnan(v)]).sum()),
                             holds_a=abs(div_a - 15) <= 1.5 and abs(c_a - 1.5) <= 0.15,
                             holds_b=abs(div_b - 15) <= 1.5 and abs(c_b - 1.5) <= 0.15))
fig3c = pd.DataFrame(rows)
print(fig3c.to_string(float_format=lambda v: f"{v:.3g}"))
print("\nreadings holding (a or b):", int((fig3c.holds_a | fig3c.holds_b).sum()), "of", len(fig3c))
print("exponent within 1.5 +- 0.15, whatever the DIV: (a)", int((fig3c.c_after_a.sub(1.5).abs() <= 0.15).sum()),
      "and (b)", int((fig3c.c_after_b.sub(1.5).abs() <= 0.15).sum()), "of", len(fig3c))
print("\nper dish, first day from which the avalanche b stays below 0.1 (inf: the last recording is at or above):")
print(pd.DataFrame(dish_rows).set_index(["dish", "sel", "F", "A"]).first_day_stays_below.unstack(["sel", "F", "A"]).to_string())

In [ ]:
# Does the outcome rest on the DIV? Under one d0 for all dishes a finite shift moves with d0 and some
# d0 puts it at 15 DIV, so only the exponent can decide; a shift never reached cannot be at 15 DIV
# under any d0. Each of the 56 readings, for shift readings (a) and (b):
long = pd.concat([fig3c[["sel", "F", "A", "d0", f"div_{k}", f"c_after_{k}"]].set_axis(
    ["sel", "F", "A", "d0", "shift_div", "c_after"], axis=1).assign(shift=k) for k in ("a", "b")])
long["shift_finite"] = np.isfinite(long.shift_div)
long["exponent_ok"] = long.c_after.sub(1.5).abs() <= 0.15
print(long.groupby(["shift", "shift_finite", "exponent_ok"]).size().to_string())
fin = long[long.shift_finite]
print("\nwith a finite shift, exponent range: %.3g to %.3g (tolerance 1.35 to 1.65)" % (fin.c_after.min(), fin.c_after.max()))
print("readings with the exponent within tolerance:")
print(long[long.exponent_ok][["sel", "F", "A", "shift", "d0", "shift_div", "c_after"]].to_string(index=False))
print("readings with a finite shift and the exponent within tolerance, under any d0:",
      int((long.shift_finite & long.exponent_ok).sum()), "of", len(long))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sizes = np.arange(1, 129)
for r, d in zip(R, C.day_index(R)):
    if r["dish"] == "23312" and r["S1"].get("aval"):
        ax[0].loglog(sizes, r["S1"]["aval"]["A1"], color=plt.cm.viridis(d / 25), lw=0.8)
ax[0].set(title="dish 23312, S1, A1, colour = day", xlabel="cascade size S", ylabel="P(S)")
h = T[(T.sel == "S1") & (T.F == "F1")]
ax[1].plot(div(h), h.avA1_b, "o", ms=3)
ax[1].axhline(0.1, color="k", ls="--")
ax[1].set(xlabel="DIV (day + 2, assumed)", ylabel="avalanche b (A1, F1)")
plt.tight_layout()

**Result: not reproduced.** Under none of the 56 readings (selection, fit, size, shift reading
and d0) are both the shift DIV and the exponent within tolerance. Under F1 the shift comes at DIV 3
to 8 for A1, too early under every d0, with median c 1.96 to 2.10; under F2 the shift is never
reached in 2 to 4 of 7 dishes. The exponent is within 1.5 plus or minus 0.15 only under S1, F2,
reading (b) (1.50 for A1, 1.49 for A2), where 4 of 7 dishes never shift, so the median shift is
later than every recording. The cell above shows that this does not rest on the DIV: wherever the
shift is reached (77 of 112 reading and shift pairs) the exponent is 1.13 to 2.53 and never within
tolerance, and the median shift is never reached in each of the 14 pairs where the exponent is
within tolerance. Under reading (b) a DIV offset changes neither the exponent nor whether the shift
is reached. Under reading (a) the same holds for any offset shared by all dishes and for the
per-dish onsets O2 and O3; other per-dish offsets were not tried, and they change the pooled moving
average and so c (S1 F1 A2 gives 1.65 under O2 and O3, against 1.72 shared), but the shift there
is at DIV 21 to 22, far from 15 plus or minus 1.5. The missing DIV therefore does not decide the
claim. The exponent rests on the claims file's reading of the fitted function, a formula lost from
the text. The round-2 and round-3 verifiers give not reproduced.


## fig3d-decay-rates-correlated

> The exponential decay rates of the variance spectrum and of the cascade size distribution are
> positively correlated across recordings, so the two power laws are established concurrently.

Criterion: one point per recording, PCA b against avalanche b, under F1, F2, A1, A2 (and S1, S2);
Spearman and Pearson both positive with two-sided p < 0.05. Per-dish values are reported.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        rs, ps = stats.spearmanr(h.b, h[f"av{A}_b"])
        rp, pp = stats.pearsonr(h.b, h[f"av{A}_b"])
        per = h.groupby("dish").apply(lambda q: stats.spearmanr(q.b, q[f"av{A}_b"])[0] if len(q) > 2 else np.nan)
        rows.append(dict(sel=sel, F=F, A=A, n=len(h), spearman=rs, p_spearman=ps, pearson=rp, p_pearson=pp,
                         dishes_positive=f"{(per > 0).sum()} of {per.notna().sum()}",
                         holds=rs > 0 and ps < 0.05 and rp > 0 and pp < 0.05))
fig3d = pd.DataFrame(rows)
print(fig3d.to_string(float_format=lambda v: f"{v:.3g}"))
print("\nshare of b at the bound 0 (below 1e-8), and the correlations without the recordings where either b is:")
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        off = h[(h.b >= 1e-8) & (h[f"av{A}_b"] >= 1e-8)]
        print(sel, F, A, "PCA b at bound %.2f, avalanche b at bound %.2f; without them n %d Spearman %.3f (p %.2g) Pearson %.3f (p %.2g)" % (
            (h.b < 1e-8).mean(), (h[f"av{A}_b"] < 1e-8).mean(), len(off),
            *stats.spearmanr(off.b, off[f"av{A}_b"]), *stats.pearsonr(off.b, off[f"av{A}_b"])))
print("\nwithout the recordings at or below 0.03 Hz mean rate (S1 keeps them when some electrodes pass):")
for (sel, F), g in by_reading(T[T.b.notna() & (T.mean_rate > C.RATE_HZ)]):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        print(sel, F, A, "n %d Spearman %.3f (p %.2g) Pearson %.3f (p %.2g)" % (
            len(h), *stats.spearmanr(h.b, h[f"av{A}_b"]), *stats.pearsonr(h.b, h[f"av{A}_b"])))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for i, F in enumerate(("F1", "F2")):
    h = T[(T.sel == "S1") & (T.F == F)].dropna(subset=["b"])
    ax[i].scatter(h.b, h.avA1_b, s=8)
    ax[i].set(xlabel="PCA b", ylabel="avalanche b (A1)", title=f"S1, {F}", xscale="symlog", xlim=(0, None))
    ax[i].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
plt.tight_layout()
print("PCA b range (S1):", T[T.sel == "S1"].b.min(), T[T.sel == "S1"].b.max())

**Result: not reproduced.** Over 112 to 122 recordings the PCA b and the avalanche b do not have
both a positive Spearman and a positive Pearson correlation at p < 0.05 under any of the 8 readings.
The Spearman correlation is 0.026 to 0.217 (p 0.022 only for S2, F1, A1, with Pearson -0.029), and
the Pearson correlation is significant only for S1, F1, A2 (0.484, Spearman 0.066, p 0.47). Per
dish, 3 or 4 of 7 dishes have a positive Spearman correlation. One of the two b's sits at the bound
0 in most recordings: the avalanche b in 0.66 to 0.71 of them under F1, the PCA b in 0.76 to 0.79
under F2. Leaving out the recordings where either is at the bound, a variant, both correlations are
positive and significant under S1, F1 (n 37, Spearman 0.501, p 0.0016, Pearson 0.456, p 0.0046, A1;
n 40, 0.602 and 0.548, A2) and under no other reading (Spearman -0.403 to -0.060).

## fig3ef-groups-cooccur and figs1-classified-by-avalanches (blocked)

fig3ef: groups by PCA b with the threshold 0.1; (a) avalanche b lower in the Power-law group
(one-sided Mann-Whitney U, p < 0.05) and its median below 0.1 in the Power-law group and at or above
0.1 in the Exponential group; (b) the cascade size distribution pooled over each group, fitted, b
below 0.1 for the Power-law group and at or above 0.1 for the Exponential group. figs1: the same with
the roles swapped (groups by avalanche b at 0.1). The group sizes show why these cannot be judged;
a median split of the PCA b is shown as a variant, not as a test of the claim.

In [ ]:
def pooled_b(results_sel, F, A):
    if not results_sel:
        return np.nan
    tot = np.zeros(C.AVALANCHE_MAX)
    n = 0
    for av_ in results_sel:
        tot += av_[A] * av_["n_cascades"]
        n += av_["n_cascades"]
    return C.fit_decay(tot / n, F, np.arange(1, C.AVALANCHE_MAX + 1.0))[1]


key_of = {(r["dish"], r["date"]): r for r in R}
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        for split, thr in (("b < 0.1", 0.1), ("median split (variant)", g.b.median())):
            pw, ex_ = g[g.b < thr], g[g.b >= thr]
            p = stats.mannwhitneyu(pw[f"av{A}_b"].dropna(), ex_[f"av{A}_b"].dropna(), alternative="less").pvalue \
                if len(pw) and len(ex_) else np.nan
            pb = [pooled_b([key_of[(d, t)][sel]["aval"] for d, t in zip(grp.dish, grp.date)], F, A) for grp in (pw, ex_)]
            rows.append(dict(sel=sel, F=F, A=A, split=split, n_power=len(pw), n_exp=len(ex_),
                             aval_b_median_power=pw[f"av{A}_b"].median(), aval_b_median_exp=ex_[f"av{A}_b"].median(),
                             mwu_p=p, pooled_b_power=pb[0], pooled_b_exp=pb[1],
                             holds_a=bool(p < 0.05 and pw[f"av{A}_b"].median() < 0.1 and ex_[f"av{A}_b"].median() >= 0.1),
                             holds_b=bool(pb[0] < 0.1 and pb[1] >= 0.1)))
fig3ef = pd.DataFrame(rows)
print(fig3ef.to_string(float_format=lambda v: f"{v:.3g}"))
print("\nfig3ef readings that hold formally at the 0.1 split:")
print(fig3ef[(fig3ef.split == "b < 0.1") & (fig3ef.holds_a | fig3ef.holds_b)][["sel", "F", "A", "n_power", "n_exp", "holds_a", "holds_b"]].to_string(index=False))
print("\nthe PCA Exponential group (b >= 0.1), by reading:")
print(T[T.b >= C.B_THRESHOLD][["sel", "F", "dish", "day", "n_kept", "mean_rate", "b", "avA1_b", "avA2_b"]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        pw, ex_ = h[h[f"av{A}_b"] < 0.1], h[h[f"av{A}_b"] >= 0.1]
        p = stats.mannwhitneyu(pw.b, ex_.b, alternative="less").pvalue if len(pw) and len(ex_) else np.nan
        rows.append(dict(sel=sel, F=F, A=A, n_power=len(pw), n_exp=len(ex_), pca_b_median_power=pw.b.median(),
                         pca_b_median_exp=ex_.b.median(), mwu_p=p,
                         pca_c_median_power=pw.c.median(), pca_c_median_exp=ex_.c.median()))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))


def pooled_spectrum_b(recs, F):
    """b of the mean of the spectra (each divided by its sum) over recordings, rank by rank, over
    ranks up to the median spectrum length (a standard choice: spectra differ in length)."""
    if not recs:
        return np.nan
    spec = [ev / ev.sum() for ev in recs]
    n = int(np.median([len(s) for s in spec]))
    mean = np.array([np.mean([s[k] for s in spec if len(s) > k]) for k in range(n)])
    return C.fit_decay(mean, F)[1]


print("\nreading (b): the variance spectra pooled over each avalanche group, fitted b")
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        pw, ex_ = h[h[f"av{A}_b"] < 0.1], h[h[f"av{A}_b"] >= 0.1]
        pb = [pooled_spectrum_b([key_of[(d, t)][sel]["ev"] for d, t in zip(grp.dish, grp.date)], F) for grp in (pw, ex_)]
        rows.append(dict(sel=sel, F=F, A=A, n_power=len(pw), n_exp=len(ex_), pooled_pca_b_power=pb[0],
                         pooled_pca_b_exp=pb[1], holds=pb[0] < 0.1 and pb[1] >= 0.1))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))

## figs2-shuffle-breaks-cooccurrence (blocked)

Readings on the shuffled data (each electrode's spike train shifted by its bin shift): (a) in a
majority of Power-law group recordings the shuffled data keep avalanche b below 0.1 while their PCA
b is at or above 0.1; (b) over all recordings the fig3d correlation on shuffled data is not positive
with p < 0.05.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b_shuf"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b_shuf"])
        pw = h[h.b < 0.1]
        a_frac = ((pw[f"av{A}_b_shuf"] < 0.1) & (pw.b_shuf >= 0.1)).mean()
        rs, ps = stats.spearmanr(h.b_shuf, h[f"av{A}_b_shuf"])
        rp, pp = stats.pearsonr(h.b_shuf, h[f"av{A}_b_shuf"])
        rows.append(dict(sel=sel, F=F, A=A, n_power=len(pw), a_fraction=a_frac,
                         shuf_aval_b_median=pw[f"av{A}_b_shuf"].median(), shuf_pca_b_median=pw.b_shuf.median(),
                         spearman=rs, p_spearman=ps, pearson=rp, p_pearson=pp,
                         a_holds=a_frac > 0.5, b_holds=not ((rs > 0 and ps < 0.05) or (rp > 0 and pp < 0.05))))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))

**Result: fig3ef-groups-cooccur, figs1-classified-by-avalanches and
figs2-shuffle-breaks-cooccurrence blocked, `methods_underspecified`.** The PCA Exponential group
holds 0 to 6 recordings, so the groups of fig3ef cannot be compared. One fit and size reading meets
both readings of fig3ef formally, S1, F1, A2: reading (a), Mann-Whitney p 0.0252 with median
avalanche b 3.7e-15 in the Power-law group and 0.543 in the Exponential group, and reading (b),
pooled b 6.1e-21 against 0.731. There the Exponential group is two near-silent recordings (23282
day 10, 16 electrodes kept, 0.005 Hz; 23317 day 0, 11 electrodes, 0.007 Hz), and only the first has
an A2 avalanche b; no other fit and size reading meets either reading, so the readings disagree. Grouped by the avalanche b, the
PCA b is lower in the avalanche Power-law group only under S1, F1 (Mann-Whitney p 0.00135, A1;
5.0e-05, A2), and the PCA b, recording by recording or pooled over each group (0.0026 to 0.0077 under
F1), is below 0.1 in both groups under every reading, so the second part of either reading cannot
hold. For figs2, reading (a) fails under every reading (at most 1.7% of recordings), while reading
(b) holds for A1 under S1 F1, S1 F2 and S2 F1 and fails for A2 and for S2 F2 A1 (shuffled Spearman
0.24 to 0.33, p 0.0003 to 0.01); the readings disagree.

## no-temporal-structure

> The cultures did not show the in vivo temporal structure of PCs, in which dominant PCs have
> longer time scales than minor PCs.

Criterion: on recordings from 15 DIV on, time scale of each PC's projection as (a) the first lag at
which its autocorrelation falls below 1/e, (b) the integral of its positive part (here, to the
first non-positive lag); the claim holds when the Spearman correlation of time scale with PC rank is
not negative with p < 0.05 in a majority of recordings. S1 and S2; d0 = 2 assumed, d0 = 1, 3, 4, 5, O2 and O3 shown.

In [ ]:
rows = []
per_rec = []
for r, d in zip(R, C.day_index(R)):
    for sel in ("S1", "S2"):
        Q = r[sel]
        if "tau_e" not in Q:
            continue
        rank = np.arange(1, len(Q["tau_e"]) + 1)
        for name in ("tau_e", "tau_e_interp", "tau_i"):
            rho, p = stats.spearmanr(rank, Q[name])
            per_rec.append(dict(dish=r["dish"], day=d, sel=sel, measure=name, rho=rho, p=p,
                                neg_sig=(rho < 0) and (p < 0.05), top10=np.median(Q[name][:10]),
                                bottom10=np.median(Q[name][-10:]), n_pc=len(rank),
                                tie_share=1 - len(np.unique(Q[name])) / len(rank),
                                q_top10=np.percentile(Q[name][:10], [10, 50, 90]),
                                q_middle=np.percentile(Q[name][10:-10], [10, 50, 90]) if len(rank) > 20 else np.full(3, np.nan),
                                q_bottom10=np.percentile(Q[name][-10:], [10, 50, 90])))
PR = pd.DataFrame(per_rec)
for (sel, name), g in PR.groupby(["sel", "measure"]):
    for d0 in D0_READINGS:
        h = g[div(g, d0) >= 15]
        rows.append(dict(sel=sel, measure=name, d0=d0, n=len(h), n_neg_sig=int(h.neg_sig.sum()),
                         frac_not_neg_sig=1 - h.neg_sig.mean(), median_rho=h.rho.median(),
                         median_top10_s=h.top10.median(), median_bottom10_s=h.bottom10.median(),
                         holds=(1 - h.neg_sig.mean()) > 0.5))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))
print("recording and measure pairs with a constant time scale (no correlation defined, counted as not negative):",
      int(PR.rho.isna().sum()), "of", len(PR))
print("\nall recordings, fraction with a significant negative correlation:")
print(PR.groupby(["sel", "measure"]).neg_sig.mean().round(3).to_string())
late = PR[div(PR) >= 15]
print("\nfrom 15 DIV (d0 = 2): share of PCs that share a value with another PC (ties), median over recordings,")
print("and the 10th, 50th and 90th percentiles of the time scale (s), median over recordings, by PC group:")
for (sel, name), g in late.groupby(["sel", "measure"]):
    qs = {k: np.nanmedian(np.vstack(g[k].values), axis=0).round(3).tolist() for k in ("q_top10", "q_middle", "q_bottom10")}
    print(f"  {sel} {name:12s} ties {g.tie_share.median():.2f}; first 10 {qs['q_top10']}; middle {qs['q_middle']}; "
          f"last 10 {qs['q_bottom10']}; median rho {g.rho.median():.3f}")

**Result: not reproduced, resting on the assumption on d0.** From 15 DIV on (d0 = 2), the time
scale falls with PC rank, Spearman negative with p < 0.05, in 25 of 44 recordings under (a), 24 of 44
under the interpolated (a'), and 27 of 44 under (b) (S1; under S2, 23, 23 and 31 of 41), so dominant
PCs have longer time scales in a majority under every d0 reading (the share without a negative
correlation is at most 0.49). The effect is small. Under (a) nearly every PC has the same value,
one bin (0.1 s; median share of tied PCs 1.00), and the median correlation is -0.118. Interpolated,
the median time scale is 0.070 s for the first 10 PCs against 0.063 s for the last 10 (median rho
-0.311); under (b) it is 0.125 s against 0.1 s (median rho -0.389).

## fig4d-cauchy-model-power-law, fig4d-gaussian-model-no-power-law (blocked)

Simulations of a binary threshold network. The Methods give theta = 1, connection probability 0.01
and the critical point g = 1, but not the network size, the scaling of the weight distributions
with g (the formula is lost from the text), the run length, the initial state or inputs, or how
model activity is binned for PCA. Not run: `methods_underspecified`.

## Summary

| Claim | Outcome |
|---|---|
| dishes-analysed | reproduced (assumed: a day with no recording does not break a run) |
| activity-onset | blocked, metadata_insufficient |
| fig1b-async-to-bursts | reproduced (assumed d0 = 2; holds for d0 = 2 to 5, O2 and O3, not d0 = 1) |
| fig1d-power-law-emerges | blocked, methods_underspecified |
| fig1e-example-dish | blocked, methods_underspecified |
| fig1f-transition-15div | blocked, methods_underspecified |
| fig1f-exponent-1p2 | blocked, methods_underspecified (depends on the transition) |
| fig1d-shuffle | blocked, methods_underspecified |
| fig2b-spatial-structure | blocked, methods_underspecified (depends on fig2c) |
| fig2c-locality-spectrum | blocked, methods_underspecified (readings disagree) |
| fig2d-locality-slope | blocked, methods_underspecified (depends on fig2c) |
| fig2d-shuffle-locality | blocked, methods_underspecified (depends on fig2c) |
| fig3c-avalanches-emerge | not reproduced |
| fig3d-decay-rates-correlated | not reproduced |
| fig3ef-groups-cooccur | blocked, methods_underspecified |
| figs1-classified-by-avalanches | blocked, methods_underspecified |
| figs2-shuffle-breaks-cooccurrence | blocked, methods_underspecified |
| no-temporal-structure | not reproduced (assumed d0 = 2) |
| fig4d-cauchy-model-power-law | blocked, methods_underspecified |
| fig4d-gaussian-model-no-power-law | blocked, methods_underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** DIV are taken as days since a dish's first
recording plus 2, an assumption (plus 1 to 5, and per-dish readings anchored on each dish's first
active recording, are shown); the paper's DIV are not in the dataset. The fitted function, the
local Moran's I and the locality slope are readings of formulas lost from the text read; the
fitted b here (rank as x) is far below the paper's 0.1 boundary in nearly every recording (median
0.0055 under S1, F1; under F2 76 to 79% of values at the bound 0, and two active recordings, 23280
day 2 and 23282 day 7, at 0.46 and 0.32), the fit ranges tried do not separate early from late
recordings, and independent Poisson counts give the same b, so the paper's b is probably computed
differently (asked of a reader). Fitting the rank as a function of
the variance, as the Figure 1D legend's axes suggest, brings b to the scale of 0.1 in early
recordings; it is shown as a variant only. Bins
start at each file's first spike, as the files give no recording start. Spikes on readout channels
with no electrode (1.2e-4 of all) are left out.

**What was found about the dataset.** The files hold no days in vitro and no plating date (they
say so), so no result can be placed on the paper's DIV axis. The time of each spike is given from
the first detected spike, not from the recording start. Seven dishes and 129 recordings are
present, with some days missing; 17 recordings have a mean rate at or below 0.03 Hz, several of
them at the end of a dish's series (23282, 23302) or in its middle (23317, days 3 and 4; 23312,
day 22). Six recordings of dish 23311 (days 8 to 13) hold 338 to 702 electrodes, and one of 23280 (day 6)
holds 525, where the others hold 956 to 1012. No
recording is deposited twice: population counts of dishes recorded at the same times correlate at
most 0.225.